<a href="https://colab.research.google.com/github/sandy-2004/weather-prediction-usingML/blob/main/weather-prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Real-Time Weather Forecasting Project
# Section 1: Import libraries

import requests
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import mean_squared_error

from datetime import datetime, timedelta


# Section 2: API configuration

API_KEY = "aae893f659ef3fe54f41c80bb7a9272e"
BASE_URL = "https://api.openweathermap.org/data/2.5/"


# Section 3: Fetch current weather from OpenWeatherMap

def get_current_weather(city_name, units="metric"):
    """
    Fetch current weather for a given city from OpenWeatherMap.
    Returns a dictionary with selected fields, or None if request fails.
    """
    url = f"{BASE_URL}weather"
    params = {
        "q": city_name,
        "appid": API_KEY,
        "units": units  # "metric" for Celsius
    }

    response = requests.get(url, params=params)
    if response.status_code != 200:
        print("Error fetching weather:", response.status_code, response.text)
        return None

    data = response.json()

    weather_info = {
        "city": data.get("name"),
        "country": data.get("sys", {}).get("country"),
        "lat": data.get("coord", {}).get("lat"),
        "lon": data.get("coord", {}).get("lon"),
        "current_temp": round(data.get("main", {}).get("temp", np.nan)),
        "feels_like": round(data.get("main", {}).get("feels_like", np.nan)),
        "temp_min": round(data.get("main", {}).get("temp_min", np.nan)),
        "temp_max": round(data.get("main", {}).get("temp_max", np.nan)),
        "humidity": round(data.get("main", {}).get("humidity", np.nan)),
        "pressure": round(data.get("main", {}).get("pressure", np.nan)),
        "clouds": data.get("clouds", {}).get("all"),
        "wind_speed": data.get("wind", {}).get("speed"),
        "wind_deg": data.get("wind", {}).get("deg"),
        "description": data.get("weather", [{}])[0].get("description"),
        "timestamp": datetime.utcfromtimestamp(data.get("dt", 0))
    }

    return weather_info



# Section 4: Read historical weather data

def read_historical_data(path="weather.csv"):
    """
    Read historical weather data from CSV.
    Expects columns like:
    MinTemp, MaxTemp, WindGustDir, WindGustSpeed, Humidity, Pressure, Temp, RainTomorrow
    """
    df = pd.read_csv(path)

    # Drop rows with missing essential values
    df = df.dropna(subset=[
        "MinTemp", "MaxTemp", "WindGustDir", "WindGustSpeed",
        "Humidity", "Pressure", "Temp", "RainTomorrow"
    ])

    # Clean column names
    df.columns = df.columns.str.strip()

    return df


# Section 5: Prepare data for classification ("Will it rain tomorrow?")

def prepare_classification_data(df):
    """
    Prepare features and labels for a classification model predicting RainTomorrow.
    Assumes target column 'RainTomorrow' with 'Yes'/'No' (or similar).
    """
    df = df.copy()

    label_encoders = {}

    # Encode target
    le_rain = LabelEncoder()
    df["RainTomorrow_enc"] = le_rain.fit_transform(df["RainTomorrow"])
    label_encoders["RainTomorrow"] = le_rain

    # Encode wind direction
    le_wind_dir = LabelEncoder()
    df["WindGustDir_enc"] = le_wind_dir.fit_transform(df["WindGustDir"])
    label_encoders["WindGustDir"] = le_wind_dir

    # Select feature columns
    feature_cols = [
        "MinTemp", "MaxTemp", "WindGustSpeed",
        "Humidity", "Pressure", "Temp", "WindGustDir_enc"
    ]
    X = df[feature_cols].values
    y = df["RainTomorrow_enc"].values

    return X, y, label_encoders, feature_cols


# Section 6: Train RandomForestClassifier for RainTomorrow

def train_rain_model(X, y):
    """
    Train a RandomForestClassifier to predict whether it will rain tomorrow.
    """
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    model = RandomForestClassifier(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    )
    model.fit(X_train, y_train)

    acc = model.score(X_test, y_test)
    print(f"Rain model accuracy: {acc:.3f}")

    return model



# Section 7: Prepare data for regression (next-hour forecasting)

def prepare_regression_data(df, feature_col):
    """
    Convert a single feature column into (X, y) pairs where:
    X = current value, y = next value.
    For example: Temp_t -> Temp_{t+1}.
    """
    values = df[feature_col].values.astype(float)

    X_list, y_list = [], []

    for i in range(len(values) - 1):
        X_list.append(values[i])
        y_list.append(values[i + 1])

    X = np.array(X_list).reshape(-1, 1)
    y = np.array(y_list)

    return X, y


# Section 8: Train RandomForestRegressor

def train_regression_model(X, y, feature_name="feature"):
    """
    Train a RandomForestRegressor on 1D time series pairs (current -> next).
    """
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    model = RandomForestRegressor(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    )
    model.fit(X_train, y_train)

    preds = model.predict(X_test)
    mse = mean_squared_error(y_test, preds)
    print(f"{feature_name} regression MSE: {mse:.3f}")

    return model


# Section 9: Predict next N steps for a feature

def predict_future_series(df, feature_col, model, steps=6):
    """
    Use a trained RandomForestRegressor to predict the next `steps` values
    for a given feature, starting from the last observed value.
    """
    last_value = float(df[feature_col].values[-1])
    predictions = []

    current_value = last_value
    for _ in range(steps):
        next_value = model.predict(np.array([[current_value]]))[0]
        predictions.append(next_value)
        current_value = next_value

    return predictions



# Section 10: Map wind degree -> compass direction

def wind_deg_to_compass(deg):
    """
    Map wind direction in degrees (0-360) to compass direction like N, NE, E, etc.
    """
    if deg is None:
        return "N"

    deg = deg % 360
    directions = [
        ("N", 0, 11.25),
        ("NNE", 11.25, 33.75),
        ("NE", 33.75, 56.25),
        ("ENE", 56.25, 78.75),
        ("E", 78.75, 101.25),
        ("ESE", 101.25, 123.75),
        ("SE", 123.75, 146.25),
        ("SSE", 146.25, 168.75),
        ("S", 168.75, 191.25),
        ("SSW", 191.25, 213.75),
        ("SW", 213.75, 236.25),
        ("WSW", 236.25, 258.75),
        ("W", 258.75, 281.25),
        ("WNW", 281.25, 303.75),
        ("NW", 303.75, 326.25),
        ("NNW", 326.25, 348.75),
        ("N", 348.75, 360)
    ]

    for name, start, end in directions:
        if start <= deg < end:
            return name
    return "N"


# Section 11: Main function tying everything together

def weather_view():
    # Step 1: city input
    city = input("Enter city name: ").strip()
    if not city:
        print("No city provided.")
        return

    # Step 2: fetch current weather
    current = get_current_weather(city)
    if current is None:
        print("Unable to get current weather.")
        return

    print("\nCurrent weather:")
    print(f"Location: {current['city']}, {current['country']}")
    print(f"Temperature: {current['current_temp']} °C")
    print(f"Feels like: {current['feels_like']} °C")
    print(f"Humidity: {current['humidity']} %")
    print(f"Pressure: {current['pressure']} hPa")
    print(f"Conditions: {current['description']}")
    print(f"Wind: {current['wind_speed']} m/s at {current['wind_deg']}°")
    print(f"Clouds: {current['clouds']} %")

    # Step 3: read historical data
    df_hist = read_historical_data("weather.csv")

    # Step 4: prepare and train classification model
    X_cls, y_cls, encoders, feature_cols = prepare_classification_data(df_hist)
    rain_model = train_rain_model(X_cls, y_cls)

    # Step 5: train regression models for Temp and Humidity
    X_temp, y_temp = prepare_regression_data(df_hist, "Temp")
    temp_reg_model = train_regression_model(X_temp, y_temp, feature_name="Temp")

    X_hum, y_hum = prepare_regression_data(df_hist, "Humidity")
    hum_reg_model = train_regression_model(X_hum, y_hum, feature_name="Humidity")

    # Step 6: build feature row from current weather for classification
    wind_dir_compass = wind_deg_to_compass(current["wind_deg"])

    le_wind = encoders["WindGustDir"]
    if wind_dir_compass in le_wind.classes_:
        wind_dir_enc = le_wind.transform([wind_dir_compass])[0]
    else:
        # fallback if unseen direction: use -1
        wind_dir_enc = -1

    feature_row = {
        "MinTemp": current["temp_min"],
        "MaxTemp": current["temp_max"],
        "WindGustSpeed": current["wind_speed"] if current["wind_speed"] is not None else 0.0,
        "Humidity": current["humidity"],
        "Pressure": current["pressure"],
        "Temp": current["current_temp"],
        "WindGustDir_enc": wind_dir_enc
    }

    X_current = np.array([[feature_row[col] for col in feature_cols]])

    rain_pred_enc = rain_model.predict(X_current)[0]
    le_rain = encoders["RainTomorrow"]
    rain_pred_label = le_rain.inverse_transform([rain_pred_enc])[0]

    print("\nPrediction:")
    print(f"Will it rain tomorrow? -> {rain_pred_label}")

    # Step 7: predict next hours for temp & humidity
    future_temp = predict_future_series(df_hist, "Temp", temp_reg_model, steps=6)
    future_hum = predict_future_series(df_hist, "Humidity", hum_reg_model, steps=6)

    print("\nNext 6 hours (approximate) temperature prediction (°C):")
    for i, val in enumerate(future_temp, start=1):
        print(f"Hour +{i}: {val:.1f}")

    print("\nNext 6 hours (approximate) humidity prediction (%):")
    for i, val in enumerate(future_hum, start=1):
        print(f"Hour +{i}: {val:.1f}")


# Section 12: Run

weather_view()

Enter city name: kolkata


/tmp/ipython-input-560657947.py:58: DeprecationWarning: datetime.datetime.utcfromtimestamp() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.fromtimestamp(timestamp, datetime.UTC).
  "timestamp": datetime.utcfromtimestamp(data.get("dt", 0))



Current weather:
Location: Kolkata, IN
Temperature: 25 °C
Feels like: 25 °C
Humidity: 41 %
Pressure: 1018 hPa
Conditions: haze
Wind: 4.63 m/s at 350°
Clouds: 0 %
Rain model accuracy: 0.822
Temp regression MSE: 13.720
Humidity regression MSE: 234.818

Prediction:
Will it rain tomorrow? -> No

Next 6 hours (approximate) temperature prediction (°C):
Hour +1: 26.1
Hour +2: 28.3
Hour +3: 26.4
Hour +4: 21.2
Hour +5: 23.8
Hour +6: 26.7

Next 6 hours (approximate) humidity prediction (%):
Hour +1: 41.9
Hour +2: 40.2
Hour +3: 45.1
Hour +4: 43.9
Hour +5: 43.8
Hour +6: 43.8
